In [ ]:
import subprocess
import cv2
import os

# Paths
darknet_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/darknet"
cfg_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/cfg/yolov4-tiny-custom.cfg"
weights_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/TRAINING/yolov4-tiny/training/yolov4-tiny-custom_best.weights"
data_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/data/obj.data"
video_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/test.mp4"
output_dir = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/output"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Initialize video capture
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Error: Unable to open video file {video_path}")
    exit(1)

# Get the frame rate (frames per second) and total frames
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps  # Duration of the video in seconds

print(f"Video FPS: {fps}, Total Frames: {total_frames}, Duration: {duration} seconds")

# Capture 1 frame every 1 second
frame_interval = fps  # 1 frame per second
saved_frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Exit the loop if no more frames are available

    # Save frame every 1 second
    if frame_count % frame_interval == 0:
        # Save the current frame
        frame_filename = os.path.join(output_dir, f"frame_{saved_frame_count:04d}.png")
        cv2.imwrite(frame_filename, frame)
        print(f"Saved frame to {frame_filename}")

        # Run Object Detection
        cmd = [
            darknet_path, 'detector', 'test', data_path, cfg_path, weights_path, frame_filename, '-thresh', '0.3'
        ]
        print(f"Running command: {' '.join(cmd)}")
        process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Print stdout and stderr to help debug
        stdout_output = process.stdout
        stderr_output = process.stderr

        print("STDOUT:\n", stdout_output)
        print("STDERR:\n", stderr_output)

        # Check and handle result image
        result_image_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/predictions.jpg"
        if os.path.exists(result_image_path):
            new_result_image_path = os.path.join(output_dir, f"frame_{saved_frame_count:04d}_predictions.jpg")
            os.rename(result_image_path, new_result_image_path)
            print(f"Moved result image to {new_result_image_path}")

            saved_frame_count += 1
        else:
            print(f"Warning: Result image not found at {result_image_path}")

    frame_count += 1

# Release video capture and print completion message
cap.release()
print(f"Processing complete. Total frames processed: {frame_count}, Total frames saved: {saved_frame_count}")